In [1]:
from experiments.bench import harness, plots
import pandas as pd
result = harness.load_result("exp3_algorithm")
print(result["environment"])

{'python': '3.12.13', 'platform': 'macOS-26.5.2-arm64-arm-64bit', 'machine': 'arm64', 'cpu_count': 10, 'cbc': 'Version: 2.10.3', 'packages': {'numpy': '2.5.1', 'scipy': '1.18.0', 'pulp': '3.3.2', 'neo4j': '6.2.0', 'tiktoken': '0.13.0', 'openai': '2.52.0', 'pandas': '3.0.5'}}


In [2]:
rows_df = pd.DataFrame(result["data"]["rows"])
display(rows_df[["algorithm", "n_people", "pair_cap", "solve_ms",
                  "optimization_ratio", "fulfillment", "unfilled"]]
        .sort_values(["n_people", "algorithm", "pair_cap"]).round(3))

violations_df = pd.DataFrame(result["data"]["violations"])
display(violations_df[["algorithm", "n_people", "pair_cap", "budget_violations"]]
        .sort_values(["n_people", "algorithm", "pair_cap"]))

print("failures (미측정 스케일):",
      [(f["n_people"], f["status"]) for f in result["data"]["failures"]])

,algorithm,n_people,pair_cap,solve_ms,optimization_ratio,fulfillment,unfilled
0,greedy,50,NaN,0.222,0.722,0.456,1
1,milp,50,1000.0,913.958,0.917,0.480,0
2,milp,50,5000.0,749.916,0.917,0.480,0
3,greedy,100,NaN,0.656,0.707,0.461,0
4,milp,100,1000.0,7371.729,0.928,0.544,0
5,milp,100,5000.0,7524.408,0.928,0.544,0
6,greedy,200,NaN,2.603,0.673,0.478,3
7,milp,200,1000.0,24664.451,0.885,0.594,0
8,milp,200,5000.0,47327.498,0.883,0.587,0


,algorithm,n_people,pair_cap,budget_violations
0,greedy,50,NaN,0
1,milp,50,1000.0,0
2,milp,50,5000.0,0
3,greedy,100,NaN,1
4,milp,100,1000.0,0
5,milp,100,5000.0,0
6,greedy,200,NaN,2
7,milp,200,1000.0,0
8,milp,200,5000.0,0


failures (미측정 스케일): [(300, 'skipped'), (500, 'skipped'), (1000, 'skipped')]


In [3]:
plots.exp3_tradeoff(result)   # figures/exp3_tradeoff.png 생성

PosixPath('/Users/honey/Dev/teamweaver/experiments/figures/exp3_tradeoff.png')

## 관찰 요약

- MILP는 측정된 모든 규모·모든 `pair_cap`(1000/5000)에서 예산 위반 0건이다. Greedy는 규모가 커질수록 위반이 늘어난다 — n=50: 0건, n=100: 1건, n=200: 2건.
- MILP의 `optimization_ratio`(cap=1000 기준)는 n=50: 0.9169, n=100: 0.9278, n=200: 0.8855 — **n=200에서 90% 목표선 아래로 떨어진다.** "≥90%" 목표는 이 실험이 기준으로 삼은 100인 데모 픽스처에서만 성립하고, 규모가 커지면 유지된다는 보장이 없다.
- `pair_cap`을 1000→5000으로 올려도 품질은 사실상 그대로다(n=200: 0.8855 vs 0.8829 — 오히려 소폭 낮음, 잡음 범위). 반면 소요 시간은 n=200에서 24.7초→47.3초로 약 1.9배 늘어난다. 즉 cap을 키우는 것은 이 규모에서 품질 이득 없이 시간만 태우는 선택이다.
- n≥300(300/500/1000)은 **미측정**이다(`data.failures`) — n=50/100/200에서 관측된 MILP(cap=1000) solve time 증가율(766.7→6933.5→22628.7ms, 배당 9.04배 → 3.26배)을 외삽하면 세션 예산을 넘어서고, n=300은 실제로 두 차례 시도 후 수렴하지 않아 중단했다(CBC 자체는 정상 동작 — 시간 내 미수렴). 대규모 구간에서 MILP가 실용적인지는 이 실험만으로는 답할 수 없다.
